In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold,RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from xgboost import XGBRFClassifier


In [86]:
df=pd.read_csv('Loan_app.csv')

In [87]:
df.head()

,Unnamed: 0,person_age,person_gender,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,loan_status
0,0,22.0,female,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,561,No,1
1,1,21.0,female,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,504,Yes,0
2,2,25.0,female,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,635,No,1
3,3,23.0,female,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,675,No,1
4,4,24.0,male,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,586,No,1


In [88]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [89]:
x=df.drop(['loan_status'],axis=1)
y=df['loan_status']

In [90]:
le=LabelEncoder()

In [91]:
x['person_gender']=le.fit_transform(x['person_gender'])

In [92]:
x['previous_loan_defaults_on_file']=le.fit_transform(x['previous_loan_defaults_on_file'])

In [93]:
x.head()

,person_age,person_gender,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file
0,22.0,0,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,561,0
1,21.0,0,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,504,1
2,25.0,0,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,635,0
3,23.0,0,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,675,0
4,24.0,1,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,586,0


In [94]:
OH=OneHotEncoder(drop='first',sparse_output=False)

In [95]:
x_enc=OH.fit_transform(df[['person_home_ownership','loan_intent']]).astype(int)

In [96]:
x_enc

array([[0, 0, 1, ..., 0, 1, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0]], shape=(45000, 8))

In [97]:
x_en=pd.DataFrame(x_enc,columns=OH.get_feature_names_out(['person_home_ownership','loan_intent']))

In [98]:
x=pd.concat([x,x_en],axis=1)

In [99]:
x.head()
x.drop(['person_home_ownership','loan_intent'],axis=1,inplace=True)

In [100]:
x.person_age=x.person_age.astype(int)

In [101]:
x.head()

,person_age,person_gender,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,0,71948.0,0,35000.0,16.02,0.49,561,0,0,0,1,0,0,0,1,0
1,21,0,12282.0,0,1000.0,11.14,0.08,504,1,0,1,0,1,0,0,0,0
2,25,0,12438.0,3,5500.0,12.87,0.44,635,0,0,0,0,0,0,1,0,0
3,23,0,79753.0,0,35000.0,15.23,0.44,675,0,0,0,1,0,0,1,0,0
4,24,1,66135.0,1,35000.0,14.27,0.53,586,0,0,0,1,0,0,1,0,0


In [102]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [103]:
Scaler=StandardScaler()

In [104]:
Scaler.fit(x_train)

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [105]:
x_train_sc=Scaler.transform(X_res)
x_test_sc=Scaler.transform(x_test)

In [106]:
lr=LogisticRegression()

In [107]:
model=lr.fit(x_train_sc,y_train)
y_pred=model.predict(x_test_sc)

In [108]:
confusion_matrix(y_test,y_pred)

array([[8165,  565],
       [ 638, 1882]])

In [109]:
accuracy_score(y_test,y_pred)

0.8930666666666667

In [110]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.93      0.94      0.93      8730
           1       0.77      0.75      0.76      2520

    accuracy                           0.89     11250
   macro avg       0.85      0.84      0.84     11250
weighted avg       0.89      0.89      0.89     11250



In [111]:
#print(classification_report(y_test,y_pred_kn))

In [112]:
svm=SVC()

In [113]:
paramgrid={
    'C':[10.0],
    'kernel':['rbf'],
    'gamma':['scale']
    
}

In [114]:
grid=GridSearchCV(estimator=svm,param_grid=paramgrid,cv=cv,n_jobs=-1,scoring='f1')

In [115]:
model=grid.fit(x_train_sc,y_train)
y_pred_kn=model.predict(x_test_sc)

In [116]:
accuracy_score(y_test,y_pred_kn)

0.9160888888888888

In [117]:
print(classification_report(y_test,y_pred_kn))

              precision    recall  f1-score   support

           0       0.93      0.96      0.95      8730
           1       0.86      0.75      0.80      2520

    accuracy                           0.92     11250
   macro avg       0.89      0.86      0.87     11250
weighted avg       0.91      0.92      0.91     11250



In [118]:
y_pred_tr=model.predict(x_train_sc)

In [119]:
accuracy_score(y_train,y_pred_tr)

0.9294222222222223

In [120]:
print(classification_report(y_test,y_pred_kn))

              precision    recall  f1-score   support

           0       0.93      0.96      0.95      8730
           1       0.86      0.75      0.80      2520

    accuracy                           0.92     11250
   macro avg       0.89      0.86      0.87     11250
weighted avg       0.91      0.92      0.91     11250



In [121]:
grid.best_params_

{'C': 10.0, 'gamma': 'scale', 'kernel': 'rbf'}

In [122]:
dt=DecisionTreeClassifier()

In [123]:
paramgrid={
    'criterion':['gini'],
    'splitter':['best'],
    'max_depth':[15],
    'min_samples_split':[15,20,25,45],
    'max_leaf_nodes':[80,85,90]
    
    
}

In [124]:
grid=GridSearchCV(estimator=dt,param_grid=paramgrid,cv=cv,n_jobs=-1,scoring='f1')

In [125]:
model=grid.fit(x_train_sc,y_train)
y_pred_dt=model.predict(x_test_sc)

In [126]:
accuracy_score(y_test,y_pred_dt)

0.9191111111111111

In [127]:
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      8730
           1       0.88      0.73      0.80      2520

    accuracy                           0.92     11250
   macro avg       0.91      0.85      0.88     11250
weighted avg       0.92      0.92      0.92     11250



In [128]:
grid.best_params_

{'criterion': 'gini',
 'max_depth': 15,
 'max_leaf_nodes': 90,
 'min_samples_split': 45,
 'splitter': 'best'}

In [129]:
y_pred_tr=model.predict(x_train_sc)
accuracy_score(y_train,y_pred_tr)

0.9295111111111111

In [130]:
print(classification_report(y_test,y_pred_kn))

              precision    recall  f1-score   support

           0       0.93      0.96      0.95      8730
           1       0.86      0.75      0.80      2520

    accuracy                           0.92     11250
   macro avg       0.89      0.86      0.87     11250
weighted avg       0.91      0.92      0.91     11250



In [154]:
from imblearn.over_sampling import SMOTE


In [164]:


sm = SMOTE()
X_res, y_res = sm.fit_resample(x_train_sc, y_train)

In [168]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek()
X_res, y_res = smt.fit_resample(x_train_sc, y_train)

In [177]:
rm=RandomForestClassifier()

In [178]:
paramgrid={
    'n_estimators':[450],
    'criterion':['log_loss'],
    #'splitter':['best','random'],
    'max_depth':[35],
    'min_samples_split':[25],
    'max_leaf_nodes':[135]
    
    
}

In [179]:
grid=GridSearchCV(estimator=rm,param_grid=paramgrid,cv=cv,n_jobs=-1,scoring='f1')

In [185]:
model=grid.fit(x_train_sc, y_train)
y_pred_rm=model.predict(x_test_sc)

In [186]:
accuracy_score(y_test,y_pred_rm)

0.9226666666666666

In [187]:
print(classification_report(y_test,y_pred_rm))

              precision    recall  f1-score   support

           0       0.93      0.98      0.95      8730
           1       0.90      0.74      0.81      2520

    accuracy                           0.92     11250
   macro avg       0.91      0.86      0.88     11250
weighted avg       0.92      0.92      0.92     11250



In [188]:
grid.best_params_

{'criterion': 'log_loss',
 'max_depth': 35,
 'max_leaf_nodes': 135,
 'min_samples_split': 25,
 'n_estimators': 450}

In [190]:
#model=grid.fit(x_train_sc,y_train)
y_pred_tr=model.predict(x_train_sc)

In [191]:
accuracy_score(y_train,y_pred_tr)

0.9308444444444445

In [192]:
print(classification_report(y_train,y_pred_tr))

              precision    recall  f1-score   support

           0       0.93      0.98      0.96     26270
           1       0.92      0.75      0.83      7480

    accuracy                           0.93     33750
   macro avg       0.93      0.87      0.89     33750
weighted avg       0.93      0.93      0.93     33750



In [ ]:
#xg=XGBRFClassifier()

In [143]:
#grid=GridSearchCV(estimator=xg,param_grid=paramgrid,cv=cv,n_jobs=-1,scoring='f1')

In [144]:
#model=grid.fit(x_train_sc,y_train)
#y_pred_xg=model.predict(x_test_sc)

In [145]:
#accuracy_score(y_test,y_pred_rm)

In [146]:
#print(classification_report(y_test,y_pred_xg))

In [147]:
df

,person_age,person_gender,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,561,No,1
1,21.0,female,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,504,Yes,0
2,25.0,female,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,635,No,1
3,23.0,female,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,675,No,1
4,24.0,male,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,645,No,1
44996,37.0,female,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,621,No,1
44997,33.0,male,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,668,No,1
44998,29.0,male,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,604,No,1


In [148]:
#df.to_csv("Loan.csv",index=False)

In [149]:
df['person_emp_exp'].unique()

array([  0,   3,   1,   5,   4,   2,   7,   6, 125,   8, 121, 101, 100,
        12,  10,   9,  14,  13,  11,  15,  16,  17,  19,  28,  25,  18,
        24,  22,  20,  23,  21,  31,  26,  27,  29,  32,  30, 124,  40,
        43,  33,  44,  34,  42,  37,  45,  36,  41,  47,  38,  39,  35,
        57,  46,  49,  48,  50,  76,  62,  61,  58,  93,  85])

In [150]:
x.head()

,person_age,person_gender,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22,0,71948.0,0,35000.0,16.02,0.49,561,0,0,0,1,0,0,0,1,0
1,21,0,12282.0,0,1000.0,11.14,0.08,504,1,0,1,0,1,0,0,0,0
2,25,0,12438.0,3,5500.0,12.87,0.44,635,0,0,0,0,0,0,1,0,0
3,23,0,79753.0,0,35000.0,15.23,0.44,675,0,0,0,1,0,0,1,0,0
4,24,1,66135.0,1,35000.0,14.27,0.53,586,0,0,0,1,0,0,1,0,0


In [151]:
df.head()

,person_age,person_gender,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,561,No,1
1,21.0,female,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,504,Yes,0
2,25.0,female,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,635,No,1
3,23.0,female,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,675,No,1
4,24.0,male,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,586,No,1


In [198]:
age=int(input("Enter age of user : "))
gender=int(input("Enter gender 1 for male 0 for female of user : "))
income=int(input("Enter income of user : "))
exp=int(input("Enter exp of user : "))
ownership=int(input("Enter ownership 0 for rent 1 for own 2 for mortgage 3 for other of user : "))
loan_amnt=int(input("Enter loan_amnt of user : "))
loan_intent=int(input("Enter loan_intent 0 for personal 1 for education 2 for medical 3 for venture 4 for homeimprovement 5 for debtconsoloidation of user : "))
loan_int_rate=int(input("Enter loan_int_rate of user : "))
loan_percent_income=float(input("Enter loan_percent_income of user : "))
credit_score=int(input("Enter credit_score of user : "))
previous_loan_defaults_on_file=int(input("Enter previous_loan_defaults_on_file 0 for no and 1 for yes of user : "))

if gender==0:
    gen='female'
elif gender == 1:
    gen='male'
else:
    print("Enter valid number")


if ownership == 0:
    own='RENT'
elif ownership == 1:
    own='OWN'
elif ownership ==2:
    own='MORTGAGE'
elif ownership == 3:
    own='OTHER'
else:
    print("Enter valid number")

if loan_intent==0:
    loan='PERSONAL'
elif loan_intent==1:
    loan='EDUCATION'
elif loan_intent==2:
    loan='MEDICAL'
elif loan_intent==3:
    loan='VENTURE'
elif loan_intent==4:
    loan='HOMEIMPROVEMENT'
elif loan_intent==5:
    loan='DEBTCONSOLIDATION'
else:
    print("Enter valid number")

if previous_loan_defaults_on_file ==0:
    prev='No'
elif previous_loan_defaults_on_file==1:
    prev='Yes'

use=pd.DataFrame([[age,gen,income,exp,own,loan_amnt,loan,loan_int_rate,loan_percent_income,credit_score,prev]],columns=['person_age', 'person_gender', 'person_income', 'person_emp_exp',
       'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate',
       'loan_percent_income', 'credit_score', 'previous_loan_defaults_on_file'])

Enter age of user :  22
Enter gender 1 for male 0 for female of user :  1
Enter income of user :  30000
Enter exp of user :  3
Enter ownership 0 for rent 1 for own 2 for mortgage 3 for other of user :  0
Enter loan_amnt of user :  20000
Enter loan_intent 0 for personal 1 for education 2 for medical 3 for venture 4 for homeimprovement 5 for debtconsoloidation of user :  0
Enter loan_int_rate of user :  10
Enter loan_percent_income of user :  0.49
Enter credit_score of user :  600
Enter previous_loan_defaults_on_file 0 for no and 1 for yes of user :  0


In [199]:
use

,person_age,person_gender,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,credit_score,previous_loan_defaults_on_file
0,22,male,30000,3,RENT,20000,PERSONAL,10,0.49,600,No


In [193]:
df['person_home_ownership'].unique()

array(['RENT', 'OWN', 'MORTGAGE', 'OTHER'], dtype=object)

In [194]:
df['loan_intent'].unique()

array(['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT',
       'DEBTCONSOLIDATION'], dtype=object)

In [196]:
df.columns

Index(['person_age', 'person_gender', 'person_income', 'person_emp_exp',
       'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate',
       'loan_percent_income', 'credit_score', 'previous_loan_defaults_on_file',
       'loan_status'],
      dtype='object')